# 📕 Pre-Workshop Notebook 6: Putting It All Together — Your Complete Research Workflow
## Scientific Workflows for Brain and Behavioral Research
### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)

---

## 📋 Final Portfolio Setup
*Run the cell below to generate your portfolio registration header.*

In [3]:
# FINAL PORTFOLIO SETUP
student_name = "Joey Johnson"
institution = "University of Missouri"
completion_date = "2026-07-20"

print(f"🎓 Portfolio record generated for: {student_name} ({institution})")
print(f"📅 Date: {completion_date}")
print("✅ Pre-Workshop Notebooks 01 through 05 confirmed complete.")

🎓 Portfolio record generated for: Joey Johnson (University of Missouri)
📅 Date: 2026-07-20
✅ Pre-Workshop Notebooks 01 through 05 confirmed complete.


---

## 🤖 Using AI Tools to Build Your Workflow ("Ask → Build → Document")

As a **Workflow Designer** completing your capstone integration, your final challenge is making sure all four stages of your workflow connect properly. Individual stages designed separately sometimes pass data in formats that the next stage does not expect — causing silent errors that are hard to spot.

When using the **Ask → Build → Document** cycle in this notebook, ask the AI to add clear status messages between each stage so you can confirm that data is being passed correctly from one step to the next.

---
## 🧠 Part A: Building a Complete Four-Stage Research Workflow
### Topic: Connecting All Stages from Data Loading to Visualization

This notebook brings together everything from Notebooks 1 through 4. You will build a single integrated workflow that connects all four stages — Obtaining Data, Analyzing Data, Visualizing Results, and Documenting Research — into one complete, working pipeline.

### 🧭 Activity A.1 — The Complete Integrated Research Workflow

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it.*
>
> "Act as a lead neuroscience workflow designer helping a research team build their first complete end-to-end analysis pipeline. Write a single Python script that connects four completely independent, clearly separated stages into one working workflow. The script should:
> 1. Stage 1 — Obtaining Data: Generate a simulated 64-channel brain recording with realistic background noise and some sudden movement artifacts. This stage should not perform any analysis or create any plots.
> 2. Stage 2 — Analyzing Data: Accept the raw data from Stage 1 and clean it using an automated threshold to remove the movement artifacts, without modifying the original data. Then extract key summary features from the cleaned signal.
> 3. Stage 3 — Visualizing Results: Accept the processed features from Stage 2 and create a high-quality figure showing the raw versus cleaned signal side-by-side, along with a summary of the extracted features.
> 4. Stage 4 — Documenting Research: Record the analysis settings used, the processing time for each stage, and any errors that were caught — saving this information to a structured log file.
> Add error-handling safeguards around each stage so that if one stage encounters a problem, it logs the issue and the other stages can still run. Include clear status messages between stages confirming that data passed correctly from one stage to the next."

In [4]:
# 💻 Activity A.1 — Paste your AI-generated Complete Research Workflow code here:

"""
End-to-End Neuroscience Analysis Pipeline
==========================================

Four independent, clearly separated stages wired into one workflow:

  1. obtain_data()        — simulate raw 64-channel recording (no analysis/plots)
  2. analyze_data()       — clean artifacts + extract features (never mutates raw)
  3. visualize_results()  — build a raw-vs-cleaned summary figure
  4. PipelineLog          — records settings, timing, and errors to a JSON log

The stages only communicate through explicit function inputs/outputs — no
stage reaches into another's internals. run_pipeline() orchestrates them,
wraps each in its own try/except so a failure in one stage doesn't stop the
others from running, and prints status messages confirming data handoff.
"""

import json
import logging
import time
import traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Output locations
# ---------------------------------------------------------------------------
OUTPUT_DIR = Path("pipeline_output")
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_PATH = OUTPUT_DIR / "pipeline_log.json"
FIGURE_PATH = OUTPUT_DIR / "results_figure.png"

# ---------------------------------------------------------------------------
# Console logging (human-readable status messages between stages)
# ---------------------------------------------------------------------------
logger = logging.getLogger("neuro_pipeline")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("[%(asctime)s] %(levelname)s: %(message)s", "%H:%M:%S"))
    logger.addHandler(handler)


class PipelineLog:
    """
    Stage 4 helper — accumulates structured metadata about the run
    (settings used, per-stage timing, per-stage status, caught errors)
    and writes it to a JSON file.
    """

    def __init__(self):
        self.record = {
            "run_timestamp": datetime.now().isoformat(),
            "settings": {},
            "stage_timing_seconds": {},
            "stage_status": {},
            "errors": [],
        }

    def set_setting(self, key, value):
        self.record["settings"][key] = value

    def log_timing(self, stage_name, seconds):
        self.record["stage_timing_seconds"][stage_name] = round(seconds, 4)

    def log_status(self, stage_name, status):
        self.record["stage_status"][stage_name] = status

    def log_error(self, stage_name, exc):
        self.record["errors"].append({
            "stage": stage_name,
            "error_type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
        })

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.record, f, indent=2)


# ===========================================================================
# STAGE 1 — Obtaining Data
# ===========================================================================
def obtain_data(n_channels=64, duration_sec=10, fs=250, artifact_rate=0.02, seed=42):
    """
    Simulate a raw multi-channel brain recording.

    - Background: a mix of low-frequency oscillations (theta/alpha/beta-like)
      plus Gaussian noise on every channel, so it looks like plausible EEG.
    - Artifacts: sudden, high-amplitude bursts affecting a random subset of
      channels at random times, mimicking movement artifacts.

    Performs NO cleaning, feature extraction, or plotting — purely generates
    and returns raw data.
    """
    rng = np.random.default_rng(seed)
    n_samples = int(duration_sec * fs)
    t = np.arange(n_samples) / fs

    band_freqs = [4, 10, 20]  # roughly theta / alpha / beta
    signal = np.zeros((n_channels, n_samples))
    for ch in range(n_channels):
        base = sum(
            rng.uniform(0.5, 1.5) * np.sin(2 * np.pi * f * t + rng.uniform(0, 2 * np.pi))
            for f in band_freqs
        )
        noise = rng.normal(0, 1.0, n_samples)
        signal[ch] = base + noise

    n_artifact_events = max(1, int(artifact_rate * n_samples))
    artifact_onsets = rng.choice(n_samples, size=n_artifact_events, replace=False)
    for onset in artifact_onsets:
        n_affected = rng.integers(5, n_channels + 1)
        affected_channels = rng.choice(n_channels, size=n_affected, replace=False)
        burst_len = rng.integers(5, 15)
        end = min(onset + burst_len, n_samples)
        amplitude = rng.uniform(15, 30) * rng.choice([-1, 1])
        signal[affected_channels, onset:end] += amplitude

    raw_data = {
        "signal": signal,               # shape: (n_channels, n_samples)
        "time": t,
        "fs": fs,
        "n_channels": n_channels,
        "artifact_onset_samples": artifact_onsets.tolist(),
    }
    return raw_data


# ===========================================================================
# STAGE 2 — Analyzing Data
# ===========================================================================
def analyze_data(raw_data, threshold_std=5.0):
    """
    Clean the raw signal and extract summary features.

    Cleaning: per-channel z-score thresholding flags samples that deviate
    more than `threshold_std` standard deviations from that channel's mean;
    flagged samples are replaced via linear interpolation from neighboring
    clean samples. The original raw signal array is never modified — a
    fresh copy is cleaned instead.
    """
    signal = raw_data["signal"]
    fs = raw_data["fs"]

    cleaned = signal.copy()  # guarantee raw_data["signal"] is untouched

    channel_mean = cleaned.mean(axis=1, keepdims=True)
    channel_std = cleaned.std(axis=1, keepdims=True)
    channel_std[channel_std == 0] = 1e-9  # avoid divide-by-zero on flat channels
    z_scores = (cleaned - channel_mean) / channel_std
    artifact_mask = np.abs(z_scores) > threshold_std

    for ch in range(cleaned.shape[0]):
        bad = artifact_mask[ch]
        if bad.any():
            good_idx = np.where(~bad)[0]
            bad_idx = np.where(bad)[0]
            if len(good_idx) > 1:
                cleaned[ch, bad_idx] = np.interp(bad_idx, good_idx, cleaned[ch, good_idx])

    features = {
        "mean_amplitude_per_channel": cleaned.mean(axis=1).tolist(),
        "std_amplitude_per_channel": cleaned.std(axis=1).tolist(),
        "rms_per_channel": np.sqrt((cleaned ** 2).mean(axis=1)).tolist(),
        "overall_mean": float(cleaned.mean()),
        "overall_std": float(cleaned.std()),
        "n_artifact_samples_removed": int(artifact_mask.sum()),
        "threshold_std_used": threshold_std,
    }

    processed = {
        "raw_signal": signal,        # original, unmodified — passed through for Stage 3
        "cleaned_signal": cleaned,
        "time": raw_data["time"],
        "fs": fs,
        "features": features,
    }
    return processed


# ===========================================================================
# STAGE 3 — Visualizing Results
# ===========================================================================
def visualize_results(processed, save_path):
    """
    Build a figure comparing raw vs. cleaned signal (for the channel most
    affected by artifact removal) alongside a feature summary panel.
    Saves the figure to `save_path` and returns that path.
    """
    raw = processed["raw_signal"]
    cleaned = processed["cleaned_signal"]
    t = processed["time"]
    features = processed["features"]

    # Highlight the channel where cleaning made the biggest difference
    diff = np.abs(raw - cleaned).sum(axis=1)
    ch = int(np.argmax(diff))

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    axes[0, 0].plot(t, raw[ch], linewidth=0.6, color="tab:red")
    axes[0, 0].set_title(f"Raw Signal — Channel {ch}")
    axes[0, 0].set_xlabel("Time (s)")
    axes[0, 0].set_ylabel("Amplitude (µV)")

    axes[0, 1].plot(t, cleaned[ch], linewidth=0.6, color="tab:blue")
    axes[0, 1].set_title(f"Cleaned Signal — Channel {ch}")
    axes[0, 1].set_xlabel("Time (s)")
    axes[0, 1].set_ylabel("Amplitude (µV)")

    axes[1, 0].bar(range(len(features["rms_per_channel"])), features["rms_per_channel"], color="tab:green")
    axes[1, 0].set_title("RMS Amplitude per Channel (cleaned)")
    axes[1, 0].set_xlabel("Channel")
    axes[1, 0].set_ylabel("RMS (µV)")

    axes[1, 1].axis("off")
    summary_text = (
        f"Overall mean:        {features['overall_mean']:.3f} µV\n"
        f"Overall std:         {features['overall_std']:.3f} µV\n"
        f"Artifact samples removed: {features['n_artifact_samples_removed']}\n"
        f"Threshold used:      {features['threshold_std_used']} std"
    )
    axes[1, 1].text(0.05, 0.5, summary_text, fontsize=12, va="center", family="monospace")
    axes[1, 1].set_title("Feature Summary")

    fig.suptitle("Neuroscience Pipeline — Raw vs Cleaned Signal & Extracted Features", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    return str(save_path)


# ===========================================================================
# ORCHESTRATION — runs all 4 stages with error isolation + Stage 4 logging
# ===========================================================================
def run_pipeline():
    plog = PipelineLog()
    settings = {
        "n_channels": 64,
        "duration_sec": 10,
        "fs": 250,
        "artifact_rate": 0.02,
        "seed": 42,
        "artifact_threshold_std": 5.0,
    }
    for key, value in settings.items():
        plog.set_setting(key, value)

    raw_data, processed, fig_path = None, None, None

    # ---- Stage 1: Obtaining Data ----
    logger.info("Stage 1 (Obtaining Data): starting...")
    t0 = time.time()
    try:
        raw_data = obtain_data(
            n_channels=settings["n_channels"],
            duration_sec=settings["duration_sec"],
            fs=settings["fs"],
            artifact_rate=settings["artifact_rate"],
            seed=settings["seed"],
        )
        plog.log_status("stage_1_obtain_data", "success")
        logger.info(
            f"Stage 1 complete: generated {raw_data['n_channels']} channels x "
            f"{raw_data['signal'].shape[1]} samples. Handing raw data to Stage 2."
        )
    except Exception as e:
        plog.log_status("stage_1_obtain_data", "failed")
        plog.log_error("stage_1_obtain_data", e)
        logger.error(f"Stage 1 failed: {e}")
    finally:
        plog.log_timing("stage_1_obtain_data", time.time() - t0)

    # ---- Stage 2: Analyzing Data ----
    if raw_data is not None:
        logger.info("Stage 2 (Analyzing Data): raw data received from Stage 1. Starting...")
        t0 = time.time()
        try:
            processed = analyze_data(raw_data, threshold_std=settings["artifact_threshold_std"])
            plog.log_status("stage_2_analyze_data", "success")
            logger.info(
                f"Stage 2 complete: removed {processed['features']['n_artifact_samples_removed']} "
                "artifact samples and extracted features. Handing processed data to Stage 3."
            )
        except Exception as e:
            plog.log_status("stage_2_analyze_data", "failed")
            plog.log_error("stage_2_analyze_data", e)
            logger.error(f"Stage 2 failed: {e}")
        finally:
            plog.log_timing("stage_2_analyze_data", time.time() - t0)
    else:
        plog.log_status("stage_2_analyze_data", "skipped (no input from Stage 1)")
        logger.warning("Stage 2 skipped: no raw data available from Stage 1.")

    # ---- Stage 3: Visualizing Results ----
    if processed is not None:
        logger.info("Stage 3 (Visualizing Results): processed features received from Stage 2. Starting...")
        t0 = time.time()
        try:
            fig_path = visualize_results(processed, FIGURE_PATH)
            plog.log_status("stage_3_visualize_results", "success")
            logger.info(f"Stage 3 complete: figure saved to {fig_path}.")
        except Exception as e:
            plog.log_status("stage_3_visualize_results", "failed")
            plog.log_error("stage_3_visualize_results", e)
            logger.error(f"Stage 3 failed: {e}")
        finally:
            plog.log_timing("stage_3_visualize_results", time.time() - t0)
    else:
        plog.log_status("stage_3_visualize_results", "skipped (no input from Stage 2)")
        logger.warning("Stage 3 skipped: no processed data available from Stage 2.")

    # ---- Stage 4: Documenting Research ----
    logger.info("Stage 4 (Documenting Research): saving settings, timing, and error log...")
    t0 = time.time()
    try:
        plog.log_status("stage_4_documenting_research", "success")
        plog.log_timing("stage_4_documenting_research", time.time() - t0)
        plog.save(LOG_PATH)
        logger.info(f"Stage 4 complete: log saved to {LOG_PATH}.")
    except Exception as e:
        plog.log_status("stage_4_documenting_research", "failed")
        plog.log_error("stage_4_documenting_research", e)
        logger.error(f"Stage 4 failed to save log: {e}")
        try:
            plog.save(LOG_PATH)  # best-effort attempt to persist whatever we have
        except Exception:
            pass

    return plog.record


if __name__ == "__main__":
    result = run_pipeline()
    print("\nPipeline run summary:")
    print(json.dumps(result["stage_status"], indent=2))

[19:32:12] INFO: Stage 1 (Obtaining Data): starting...
INFO:neuro_pipeline:Stage 1 (Obtaining Data): starting...
[19:32:12] INFO: Stage 1 complete: generated 64 channels x 2500 samples. Handing raw data to Stage 2.
INFO:neuro_pipeline:Stage 1 complete: generated 64 channels x 2500 samples. Handing raw data to Stage 2.
[19:32:12] INFO: Stage 2 (Analyzing Data): raw data received from Stage 1. Starting...
INFO:neuro_pipeline:Stage 2 (Analyzing Data): raw data received from Stage 1. Starting...
[19:32:12] INFO: Stage 2 complete: removed 478 artifact samples and extracted features. Handing processed data to Stage 3.
INFO:neuro_pipeline:Stage 2 complete: removed 478 artifact samples and extracted features. Handing processed data to Stage 3.
[19:32:12] INFO: Stage 3 (Visualizing Results): processed features received from Stage 2. Starting...
INFO:neuro_pipeline:Stage 3 (Visualizing Results): processed features received from Stage 2. Starting...
[19:32:12] INFO: Stage 3 complete: figure saved


Pipeline run summary:
{
  "stage_1_obtain_data": "success",
  "stage_2_analyze_data": "success",
  "stage_3_visualize_results": "success",
  "stage_4_documenting_research": "success"
}


 ### ✍️ Part A Reflection


 <!--
*Double-click this cell to write your response.*

* **What I observed:** Look at the status messages printed between stages. Did each stage receive data in the format it expected? Were any errors caught and logged?
* **Connecting to key concepts:** This integrated workflow demonstrates mastery of all 50 concepts in the Reference Guide. In your own words, describe what happens if Stage 2 (Analyzing Data) tries to pass its results to Stage 3 (Visualizing Results) in the wrong format — and how the error-handling safeguards in the workflow help you find this quickly. -->



Running the script, the status messages confirmed each handoff worked as expected:

```
Stage 1 complete: generated 64 channels x 2500 samples. Handing raw data to Stage 2.
Stage 2 complete: removed 478 artifact samples and extracted features. Handing processed data to Stage 3.
Stage 3 complete: figure saved to pipeline_output/results_figure.png.
Stage 4 complete: log saved to pipeline_output/pipeline_log.json.
```

Each stage received data in exactly the shape the next stage expected:

- **Stage 1** returned a dict with `signal`, `time`, and `fs`.
- **Stage 2** unpacked those by key, cleaned a *copy* of the signal, and returned a new dict (`raw_signal`, `cleaned_signal`, `time`, `fs`, `features`).
- **Stage 3** unpacked *that* dict by key to build the figure.

No errors were caught on this run — the `errors` list in `pipeline_log.json` came back empty, and every stage's status logged as `"success"`. The final JSON log also captured the settings used (64 channels, 250 Hz, 5-std threshold) and the per-stage timing, so the full run is reproducible and auditable after the fact.

## Connecting to Key Concepts

The stages are connected by a **contract**, not by shared variables — Stage 3 doesn't know or care *how* Stage 2 computed `cleaned_signal`, it only cares that the dict it receives has that key, plus `raw_signal`, `time`, and `features` in the shape it expects (e.g., `features["rms_per_channel"]` being a list it can index and plot).

**If Stage 2 broke that contract** — say it renamed `cleaned_signal` to `clean_signal`, returned `features` as a single number instead of a dict, or accidentally passed back the raw signal instead of the cleaned one — Stage 3 would fail the moment it tried to access the missing or mismatched key, most likely with a `KeyError` or a `TypeError` when it tried to plot something that wasn't the array shape it expected.

**That's exactly what the `try/except` block around Stage 3 is for.** Instead of the whole script crashing with a raw traceback dumped to the console and no record of what happened:

1. The exception is **caught**, not left to crash the program.
2. The stage's status is **logged as `"failed"`**.
3. The full error type, message, and traceback are **written into `pipeline_log.json`** under that stage's entry.
4. **Stage 4 still runs** and writes the log, even though Stage 3 failed.

That combination — *isolate the failure → record exactly where and why it happened → let everything else keep running* — is what lets me open the log file afterward and immediately see "Stage 3 failed, here's the traceback," rather than having to re-run the whole pipeline with a debugger to figure out which stage broke and why.

---

## 📑 You Are Ready for the Onsite Workshop

Completing this notebook means you have worked through all six foundational areas:

1. **Notebook 1:** How brain and behavioral data is captured and how computers organize it
2. **Notebook 2:** Working with larger datasets and running analyses efficiently
3. **Notebook 3:** Accessing national research computing resources and connecting securely
4. **Notebook 4:** Cleaning signals and building reliable workflows
5. **Notebook 5:** Finding patterns and making workflows reproducible
6. **Notebook 6:** Connecting all four stages into a complete integrated workflow

The onsite workshop will apply these foundations across three progressively more demanding scenarios — a single-participant workflow, a group of 50 participants, and finally a small portable edge device. Everything you have practiced here will be directly relevant.


---

## 📖 Reference Guide: 50 Key Concepts for Neuroscience Workflows

Keep this reference guide handy throughout all six pre-workshop notebooks and during the onsite labs. These concepts form the foundation of the workshop's hands-on activities.

### 🧠 Neuroscience & Research Applications (Concepts 1–25)
1. **Brain-Machine Interface (BMI):** A system that connects the brain directly to an external device — bypassing damaged nerves or muscles — so that brain signals can control a robotic limb, cursor, or other tool.
2. **Non-Invasive (EEG) vs. Invasive (Intracortical) Sensors:** Scalp electrodes (EEG) are easy to use but pick up blurry, averaged signals through the skull. Implanted microelectrode arrays record from individual neurons with much greater precision, but require surgery.
3. **Signal Delay (Latency):** The time gap between a brain signal being recorded and a device responding to it. Delays longer than about 50–100 milliseconds feel unnatural to the user of a prosthetic device.
4. **Decoder:** A mathematical or statistical model that translates continuous brain signal patterns into a useful output — such as movement direction, cursor position, or speech intent.
5. **fMRI:** Functional Magnetic Resonance Imaging — a brain scanning method that measures blood oxygen levels as a stand-in for neural activity. When neurons become active, they consume more oxygen, causing a detectable change in the local blood signal.
6. **Voxel:** A small 3D cube of brain tissue in an fMRI image — the brain-imaging equivalent of a pixel. Each voxel contains millions of neurons.
7. **Head Movement Correction:** A processing step that aligns brain scan images across time, compensating for small movements the participant made during the recording session.
8. **Open-Loop vs. Closed-Loop Systems:** An open-loop device follows a fixed program regardless of what the user's brain is doing. A closed-loop device continuously reads incoming signals and adjusts its output in real time based on what it detects.
9. **Deep Brain Stimulation (DBS):** A treatment for conditions like Parkinson's disease in which a small implanted device delivers electrical pulses to specific brain regions to reduce tremor and improve movement.
10. **EMG (Electromyogram):** A recording of the electrical signals produced by muscles when they contract. Often used alongside brain recordings to verify whether a behavioral response actually occurred.
11. **Continuous Recordings vs. Event Markers:** A continuous recording captures an uninterrupted time series of measurements. Event markers are specific timestamps that label when something important happened — such as when a stimulus appeared or a button was pressed.
12. **Signal Drift:** The gradual change in a recorded signal over time, not due to the brain, but due to electrode movement, tissue changes, or electronic drift. Workflows need to account for this to keep analysis accurate over long sessions.
13. **Open Data Repositories:** Publicly accessible online archives (such as DANDI, OpenNeuro, or the Human Connectome Project) where researchers share their raw data so others can analyze or replicate their findings.
14. **Automated Data Download (API):** A method of downloading data directly inside a script using a standardized web link, rather than clicking through a website manually. This makes data access reproducible and audit-able.
15. **Artifact Removal:** The process of identifying and removing unwanted signals — such as electrical noise from the building, muscle movements, or eye blinks — from a raw brain recording before analysis.
16. **Downsampling:** Reducing the number of data points per second in a recording, to save memory and processing time, when the extra detail is not needed for the analysis.
17. **Standard Data Formats (BIDS):** A community-agreed system for naming and organizing brain imaging files so that any researcher or software tool can understand the structure without needing special instructions.
18. **Spectrogram:** A visual display showing how the frequency content of a signal changes over time — useful for seeing when the brain shifts between different rhythmic states such as sleep stages or attention levels.
19. **Nyquist Rule:** A fundamental rule of digital recording: to accurately capture a signal, you must record at least twice as fast as the highest frequency in that signal. Recording too slowly creates false patterns called aliasing.
20. **Local Field Potential (LFP):** An electrical recording that reflects the combined activity of a small cluster of nearby neurons — capturing the general activity level of a local brain region rather than individual cells.
21. **Signal Transfer Function:** A mathematical description of how an input signal is transformed into an output signal by a processing step — useful for predicting what a filter or decoder will do to any given input.
22. **Spike Sorting:** The process of separating a mixed electrical recording from multiple nearby neurons into individual neuron signals, based on the distinct shape of each neuron's electrical discharge.
23. **Machine Learning for Neural Decoding:** Using statistical learning algorithms to automatically find patterns in brain signal data that predict behavior, movement intent, or cognitive state.
24. **Sensory Feedback:** Sending information back to the user of a brain-machine interface — for example, delivering a gentle electrical sensation to the skin to simulate the feeling of touching an object with a prosthetic hand.
25. **Neural Plasticity:** The brain's ability to reorganize its connections over time. Relevant to BMI research because users can learn to control devices more accurately with practice as their brain adapts.

### 💻 Computing & Workflow Fundamentals (Concepts 26–50)
26. **Working Memory (RAM) vs. Permanent Storage:** RAM (working memory) holds data only while the computer is on — it is extremely fast but temporary. The hard drive stores files permanently but is much slower to access.
27. **Processor Core:** A single computing unit inside a central processor (CPU). Most modern computers have multiple cores, allowing several tasks to run at the same time.
28. **CPU vs. GPU:** A CPU handles complex, varied tasks one at a time across a few powerful cores. A GPU handles simple, repetitive tasks across thousands of smaller cores simultaneously — useful for image processing and machine learning.
29. **Memory Overflow:** What happens when a script tries to load more data into working memory (RAM) than the computer has available — causing the program to crash.
30. **Motherboard:** The main circuit board inside a computer that connects all the components — processor, memory, storage, and network — so they can communicate with each other.
31. **Processor Slowdown (Thermal Throttling):** When a processor gets too hot, it automatically slows itself down to prevent damage. This can cause unexpected slowdowns during long analysis runs.
32. **High-Speed Processor Cache:** A small, extremely fast memory area built directly into the processor, used to store frequently needed values so they do not have to be fetched from RAM repeatedly.
33. **Local vs. Cloud Computing:** Running your analysis on the computer in front of you (local) versus running it on a remote server accessed over the internet (cloud). Cloud computing allows access to much more memory and processing power.
34. **Temporary Cloud Workspace:** Cloud computing environments like Google Colab provide a temporary workspace that is automatically cleared when you close the session. Any files you need to keep must be saved to permanent storage before the session ends.
35. **Internet Speed as a Bottleneck:** When downloading large research datasets, the speed of your internet connection often limits how fast data can arrive — regardless of how fast your computer itself is.
36. **Organizing Code into Stages:** Separating a workflow into clearly defined, independent stages — such as one stage for loading data, one for analysis, and one for visualization — makes it much easier to find and fix problems.
37. **Whole Numbers vs. Decimal Numbers in Computing:** Computers store whole numbers (integers) very efficiently. Decimal numbers (floating-point) require more memory and processing time. Choosing the right type for your data can affect both speed and accuracy.
38. **Automated Data Access (API):** A standardized connection that allows one piece of software to request data or services from another automatically — for example, a script that downloads data from a research repository without any manual steps.
39. **Settings Files (JSON/YAML):** Lightweight text files used to store configuration settings, metadata, and parameters for a workflow — making it easy to share, reproduce, or adjust an analysis without changing the code itself.
40. **Processing Delay:** The time between when data arrives in your workflow and when your analysis produces a result. In real-time recording systems, keeping this delay short is critical.
41. **Keeping Stages Independent:** Designing a workflow so that the data analysis stage does not depend on the visualization stage, and vice versa. This means you can update or replace one stage without breaking the others.
42. **Text Files vs. Optimized Data Files:** Saving data as plain text (such as CSV) is easy to read in a spreadsheet but very slow for large datasets. Optimized formats (such as HDF5 or NumPy binary files) are much faster to load and take up less disk space.
43. **Simultaneous Processing (Parallel Computing):** Splitting a large task — such as analyzing 50 participants — into smaller chunks that run at the same time across multiple processor cores, rather than one after another.
44. **Hidden Configuration Settings:** Values that a workflow needs — such as access keys for a data repository — that are stored securely in the operating system rather than written directly into the code, to prevent accidental exposure.
45. **Software Libraries (Dependencies):** Pre-built collections of code (such as NumPy, SciPy, or scikit-learn) that provide ready-made tools for common tasks — so you do not have to write mathematical functions from scratch.
46. **Code Version Tracking (Git):** A system that records every change made to a set of code files over time, along with who made the change and when. This allows teams to collaborate and to restore earlier versions if something goes wrong.
47. **Error Handling:** Code that anticipates things going wrong — such as a missing file or a calculation that produces an undefined result — and responds gracefully rather than crashing the entire workflow.
48. **Data Array:** A structured list of numbers organized so that a computer can perform calculations on all of them efficiently — the basic building block of scientific data analysis.
49. **Data Backlog:** What happens when data arrives faster than your workflow can process it — causing a growing queue that eventually uses up available memory.
50. **Protecting Original Data:** A core rule of reproducible research: never modify your raw data files. Always write processed results to a new, separate file so the original record remains intact.


---

## ✅ Grand Portfolio Registry: Confirming Your Complete Preparation

*Check off each handbook below once you have completed its activities and filled in its self-check table.*

| Pre-Workshop Notebook | Concept Areas Covered | Status |
|---|---|---|
| **Notebook 01: Foundations** | Concepts 1–10, 26–35 | [x] Completed |
| **Notebook 02: Larger Datasets** | Concepts 11–14, 36–43 | [x] Completed |
| **Notebook 03: National Research Computing Access** | Own 15-concept FABRIC Reference Guide (not part of this list) | [x] Completed |
| **Notebook 04: Signal Processing** | Concepts 15–21, 45–47 | [x] Completed |
| **Notebook 05: Analysis & Patterns** | Concepts 22–25, 44, 48–50 | [x] Completed |
| **Notebook 06: Full Integration** | All 50 Concepts (plus the FABRIC Reference Guide from Notebook 3) | [x] Completed |